# Navigation — Banana Collector

An interactive walkthrough of the Unity Banana Collector environment and the
agents in this repository.

This notebook is deliberately **thin**: all the real logic lives in the
`banana_nav` package so the same code runs here, from the CLI, and in the
multi-seed ablation. For the full study and results, see
[Report.md](Report.md); for how the 2018 Unity environment was brought to a
modern Python stack, see [docs/PORTING.md](docs/PORTING.md).

**Prerequisites** — `pip install -e .` and the Unity build unzipped into the
repository root (see the README).

## 1. Start the environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from banana_nav.env import BananaEnv
from banana_nav.agent import AgentConfig, DQNAgent
from banana_nav.config import build, list_configs
from banana_nav.train import train, evaluate, TrainConfig

print("available configs:", list_configs())

In [ ]:
# no_graphics=True runs headless and much faster; set False to watch the window.
env = BananaEnv(no_graphics=True, seed=0)

print("State size :", env.state_size)
print("Action size:", env.action_size)
print("Brain      :", env.brain_name)

## 2. A random agent

The baseline to beat. Actions are chosen uniformly at random, which scores
around zero — yellow and blue bananas are collected in roughly equal numbers,
so the rewards cancel out.

In [ ]:
scores = []
for i in range(5):
    state = env.reset(train_mode=True)
    score, done = 0.0, False
    while not done:
        state, reward, done, _ = env.step(np.random.randint(env.action_size))
        score += reward
    scores.append(score)

print("random agent scores:", scores)
print("mean:", np.mean(scores))

## 3. Train an agent

Any config in `configs/` can be loaded by name. `rainbow` combines Double DQN,
a dueling head, prioritized replay, 3-step returns and NoisyNet exploration.

The run below is short so the notebook stays runnable — it will **not** solve
the environment. For a real run use 900+ episodes, or the CLI:

```
banana-train train --config rainbow --episodes 900
```

In [ ]:
name, agent_cfg, train_cfg = build("rainbow", env.state_size, env.action_size,
                                   overrides={"n_episodes": 50})

print(f"variant: {name}")
for k in ("double", "dueling", "prioritized", "noisy", "n_step"):
    print(f"  {k:12} = {getattr(agent_cfg, k)}")

In [ ]:
result = train(env, agent_cfg, train_cfg, variant=name, seed=0,
               checkpoint_path="checkpoints/notebook_demo.pth", verbose=True)

print("\nbest 100-episode average:", round(result.best_avg, 2))
print("solved in:", result.solved_episode, "episodes")

## 4. Plot the run

In [ ]:
scores = np.array(result.scores)
avg = np.array(result.moving_avg)
ep = np.arange(1, len(scores) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ep, scores, color="#2a78d6", alpha=0.25, linewidth=1)
ax.plot(ep, avg, color="#2a78d6", linewidth=2, label="100-episode average")
ax.axhline(13.0, color="#898781", linestyle="--", linewidth=1.2, label="solved (+13)")
ax.set_xlabel("Episode"); ax.set_ylabel("Score")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

## 5. Watch a trained agent

Load a checkpoint produced by a full training run and evaluate it **greedily** —
no epsilon, no weight noise. This is the number that decides whether the
environment is solved.

In [ ]:
from pathlib import Path

ckpt = Path("checkpoints/rainbow_seed0.pth")
if ckpt.exists():
    agent = DQNAgent.load(ckpt)
    stats = evaluate(env, agent, episodes=20, verbose=False)
    print(f"mean over 20 greedy episodes: {stats['mean']:.2f}  (min {stats['min']:.0f}, max {stats['max']:.0f})")
else:
    print(f"{ckpt} not found - run a full training job first, e.g.")
    print("  banana-train train --config rainbow --episodes 900")

## 6. Clean up

Always close the environment. If an exception escapes before `close()`, the
non-daemon gRPC server thread keeps Python alive holding a live `Banana.exe` —
which is why `BananaEnv` is a context manager (`with BananaEnv() as env:`).

In [ ]:
env.close()

## Next steps

* **Full ablation** — `banana-train ablate --seeds 5 --episodes 900 --workers 6`
* **Figures** — `banana-train plot --results results/ablation --out assets`
* **Results and discussion** — [Report.md](Report.md)